# Personalized FL for Heart Disease Risk Prediction — Colab Runner

Runs this project's experiments in Google Colab instead of the local Windows machine, which blocks several native ML library DLLs under Smart App Control (see `README.md`, "Environment note" section).

**Usage:** Runtime → Change runtime type → CPU is fine (no GPU needed; this project targets standard hardware). Then Runtime → Run all.

This notebook: (1) clones the repo, (2) installs `requirements.txt` fresh, (3) re-verifies Phase 3/4/5 still reproduce their committed results, (4) runs Phase 6's SHAP explainability analysis.

## 1. Clone the repository

In [ ]:
REPO_URL = "https://github.com/Teena2812/PBL-5.git"

import os
if os.path.exists("project"):
    %cd project
    !git pull
else:
    !git clone $REPO_URL project
    %cd project

## 2. Install dependencies

Colab already ships numpy/pandas/scikit-learn/matplotlib/torch, but we install
from `requirements.txt` to match the pinned versions used for the committed
results (pandas==2.2.3 and matplotlib==3.8.4 were pinned only to work around
the LOCAL machine's Windows Application Control block -- those specific pins
aren't needed on Colab, but installing them keeps results reproducible
against what's documented in the README).

In [ ]:
!pip install -q -r requirements.txt

## 3. Sanity check: core libraries import cleanly

In [ ]:
import numpy, pandas, sklearn, matplotlib, torch, flwr
print("numpy", numpy.__version__)
print("pandas", pandas.__version__)
print("scikit-learn", sklearn.__version__)
print("matplotlib", matplotlib.__version__)
print("torch", torch.__version__)
print("flwr", flwr.__version__)

from sklearn.ensemble import RandomForestClassifier
import numpy as np
RandomForestClassifier(n_estimators=5).fit(np.random.rand(10, 3), np.random.randint(0, 2, 10))
print("RandomForestClassifier: OK")

## 4. Re-verify Phase 3 (Local ML / Centralized ML baselines)

Should reproduce the committed results: Local ML mean accuracy ~0.833 (RF) / ~0.819 (LR), Centralized ML ~0.842 (RF) / ~0.855 (LR).

In [ ]:
!python experiments/phase3_baselines.py

## 5. Re-verify Phase 4 (FedAvg via Flower)

Should reproduce: FedAvg final global weighted accuracy ~0.829 (single seed=42 run).

In [ ]:
!python experiments/phase4_fedavg.py

In [ ]:
!python experiments/phase4_nn_baselines.py

### Optional: Phase 4/5 multi-seed robustness checks (slower, ~5-15 min each)

Only re-run these if you need to re-verify the multi-seed numbers already committed in `experiments/results/phase4_multiseed_*.csv` and `phase5_multiseed_*.csv` -- otherwise skip to Phase 6 below.

In [ ]:
# !python experiments/phase4_multiseed_comparison.py
# !python experiments/phase5_fedprox.py
# !python experiments/phase5_multiseed_comparison.py
# !python experiments/phase5_equity_analysis.py

## 6. Phase 6: SHAP explainability

This is the analysis that Smart App Control blocked locally (scikit-learn's
compiled tree extensions, then numba, then more sklearn submodules --
whack-a-mole DLL blocking). Should run cleanly on Colab.

In [ ]:
import shap
print("shap", shap.__version__, "-- import OK")

In [ ]:
!python experiments/phase6_shap_analysis.py

## 7. Display the generated charts inline

In [ ]:
from IPython.display import Image, display
import glob

for path in sorted(glob.glob("experiments/results/phase6_*.png")):
    print(path)
    display(Image(filename=path))

## 9. Phase 7 prep: save personalized model checkpoints + preprocessing artifact

Saves each hospital's final personalized model (FedProx + fine-tuning, same
pipeline as Phase 5) as a PyTorch state_dict, plus a `preprocessing.json`
capturing the standardization constants and categorical encoding needed to
transform a brand-new raw patient record at inference time. These back
Phase 7's live single-patient prediction endpoint (inference + single-instance
SHAP only -- no retraining happens there).

The printed personalized metrics should exactly match Phase 5's
`[Personalized FL]` results above, as a sanity check.

In [ ]:
!python experiments/save_personalized_models.py

In [ ]:
!zip -r phase7_models.zip experiments/results/models/
from google.colab import files
files.download("phase7_models.zip")

## 10. Phase 7: verify the FastAPI backend's live prediction endpoint

Starts the backend in the background, then sends a real request to
`POST /api/predict` -- this is the one code path
(`src/backend/inference.py`) that has never actually executed anywhere
yet, since it needs torch/shap and the local dev machine can't run those.
Everything else in the backend (the 6 dashboard data endpoints) was
already verified locally with `TORCH_AVAILABLE=False`; this cell verifies
the part that couldn't be.

In [ ]:
import subprocess
import time

import requests

backend_proc = subprocess.Popen(
    ["uvicorn", "src.backend.main:app", "--host", "127.0.0.1", "--port", "8000"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)

# Wait for the server to come up (poll /api/health instead of a fixed sleep).
base_url = "http://127.0.0.1:8000"
for _ in range(30):
    try:
        r = requests.get(f"{base_url}/api/health", timeout=1)
        if r.status_code == 200:
            break
    except requests.exceptions.ConnectionError:
        pass
    time.sleep(1)
else:
    print(backend_proc.stdout.read())
    raise RuntimeError("Backend did not come up in time -- see server log printed above.")

print("GET /api/health ->", r.status_code, r.json())
assert r.json()["live_prediction_available"] is True, (
    "Expected live_prediction_available=True on Colab (torch/shap should be installed here)."
)

In [ ]:
import json

# Example patient (same values as schemas.py's PatientInput.example) against
# each of the 5 hospitals' personalized models, to sanity-check every
# checkpoint loads and produces a plausible response, not just one.
payload_base = {
    "age": 63, "sex": 1, "cp": 4, "trestbps": 145, "chol": 233,
    "fbs": 1, "restecg": 0, "thalach": 150, "exang": 0,
    "oldpeak": 2.3, "slope": 1, "ca": 0, "thal": 6,
}

for hospital_id in ["hospital_1", "hospital_2", "hospital_3", "hospital_4", "hospital_5"]:
    payload = {**payload_base, "hospital_id": hospital_id}
    r = requests.post(f"{base_url}/api/predict", json=payload, timeout=30)
    print(f"--- {hospital_id} ---")
    print(r.status_code)
    print(json.dumps(r.json(), indent=2))
    print()
    assert r.status_code == 200, f"Expected 200 for {hospital_id}, got {r.status_code}: {r.text}"

In [ ]:
# Error-path checks: invalid hospital_id and invalid category should both
# return clean 400s (not 500s), since transform_new_patient() raises
# ValueError for values never seen during training.
#
# thal's valid trained categories are {3, 6, 7} (non-sequential), so its
# Pydantic bound is deliberately loose (0-7) to admit all three -- an
# in-range-but-invalid value like thal=4 correctly passes Pydantic and
# reaches transform_new_patient()'s category check, giving 400. A
# clearly-out-of-bounds value like thal=99 would be caught by Pydantic
# first (422) and never reach that check at all -- a different code path,
# not what this cell is testing.

r = requests.post(f"{base_url}/api/predict", json={**payload_base, "hospital_id": "hospital_99"}, timeout=10)
print("invalid hospital_id ->", r.status_code, r.json())
assert r.status_code == 400

r = requests.post(f"{base_url}/api/predict", json={**payload_base, "thal": 4, "hospital_id": "hospital_1"}, timeout=10)
print("in-range but untrained thal=4 ->", r.status_code, r.json())
assert r.status_code == 400

# Pydantic-level validation (out-of-range field) should 422.
r = requests.post(f"{base_url}/api/predict", json={**payload_base, "age": -5, "hospital_id": "hospital_1"}, timeout=10)
print("invalid age=-5 ->", r.status_code)
assert r.status_code == 422

# And a clearly out-of-bounds thal also hits Pydantic (422), not the app-level check.
r = requests.post(f"{base_url}/api/predict", json={**payload_base, "thal": 99, "hospital_id": "hospital_1"}, timeout=10)
print("out-of-bounds thal=99 ->", r.status_code)
assert r.status_code == 422

print("\nAll error-path checks passed.")

In [ ]:
# Stop the background server.
backend_proc.terminate()
backend_proc.wait(timeout=10)
print("Backend stopped.")

## 11. Phase 7: export sample patients for the Explainability screen

Runs one held-out test patient per hospital through that hospital's saved
personalized model, using the same `predict_and_explain()` as
`POST /api/predict` (inference + KernelSHAP only, no retraining). Writes
`experiments/results/dashboard_data/sample_patients.json` so the dashboard
can show real predictions locally, where torch is blocked.

Download the file and copy it into the same path in the local repo.

In [ ]:
!python experiments/export_sample_patients.py

In [ ]:
from google.colab import files
files.download("experiments/results/dashboard_data/sample_patients.json")

## 8. Download results back to sync with the local repo

Zips `experiments/results/` so you can download it and copy the new Phase 6
outputs (CSVs + PNGs) back into the local checkout before committing.

In [ ]:
!zip -r phase6_results.zip experiments/results/phase6_*
from google.colab import files
files.download("phase6_results.zip")